<a href="https://colab.research.google.com/github/Boran-Sert/SoftITO_Projler/blob/master/Python/22_10_2026(MODEL_Y%C3%9CKLEME).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_adı = "Qwen/Qwen3.8-27B"
cihaz = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(model_adı, trust_remote_code=True)
print(f"Kullanılan cihaz: {cihaz}")
model = AutoModelForCausalLM.from_pretrained(model_adı,trust_remote_code=True).to(cihaz)
model.eval()
print(f"paramatre sayısı: {model.num_parameters()}")

config.json:   0%|          | 0.00/4.31k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/17.9k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/8.95k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/202 [00:00<?, ?B/s]

paramatre sayısı: 26895998464


In [ ]:
ornek = "Yapay zeka dünyayı değiştiriyor."
tokenler = tokenizer.tokenize(ornek)
idler = tokenizer.encode(ornek)
print(f"Tokenler: {tokenler}")
print(f"İdler: {idler}")
print(f"geri çözülmüş:", tokenizer.decode(idler))

Tokenler: ['Y', 'ap', 'ay', 'Ġz', 'eka', 'ĠdÃ¼ny', 'ayÄ±', 'ĠdeÄŁiÅŁtir', 'iyor', '.']
İdler: [56, 385, 347, 1110, 50557, 162837, 159601, 215900, 80467, 13]
geri çözülmüş: Yapay zeka dünyayı değiştiriyor.


In [ ]:
def sor(soru, sistem = "Sen yardımsever bir asistansın. Türkçe ve kısa cevap ver.",
        max_yeni_token = 2000, temperature= 0.3, top_p=0.9):
  mesajlar = [
      {"role": "system", "content": sistem},
      {"role": "user", "content": soru}
  ]

  metin = tokenizer.apply_chat_template(mesajlar, tokenize=False, add_generation_prompt=True)
  tokenler = tokenizer(metin, return_tensors="pt").to(cihaz)
  with torch.no_grad():
    cikti = model.generate(
      **tokenler,
      max_new_tokens=max_yeni_token,
      do_sample=temperature > 0,
      temperature=temperature if temperature > 0 else None,
      top_p=top_p if temperature > 0 else None,
      pad_token_id=tokenizer.eos_token_id,
    )
  yeni = cikti[0][tokenler["input_ids"].shape[1]:]
  return tokenizer.decode(yeni, skip_special_tokens=True)

print(sor("Nabers"))

We need to respond to user: "Nabers". User says in Turkish: "Sen yardımsever bir asistansın. Türkçe ve kısa cevap ver." Means: You are a helpful assistant. Answer in Turkish and short. User: "Nabers" likely informal "Naber?" (what's up). Need final in Turkish, short. Need final concise.
</think>

İyiyim, teşekkürler! Sen nasılsın?


In [ ]:
sorular = [
    "Fatih 3'ü bir ada sever mi ?",
    "Komşular iyi midir ?",
    "Maria mı haklıydı yoks Ragıp Efendi mi ?",
    "Evlenmeden önce kaç yıl sevgili kalınmalıdır ?"
]
for s in sorular:
  print(f"? {s}\n {sor(s)}\n " + ":(\n"*4)

? Fatih 3'ü bir ada sever mi ?
 Thinking Process:

1.  **Analyze the Request:**
    *   User asks: "Fatih 3'ü bir ada sever mi ?" (Does Fatih 3 love an island? / Does Fatih 3 like an island?)
    *   Language: Turkish.
    *   Constraint 1: Helpful assistant.
    *   Constraint 2: Turkish and short answer.
    *   Constraint 3 (System): Neutrality on political topics (not applicable here, "Fatih 3" is likely a person, a pet, or a fictional character, not a political topic).

2.  **Analyze the Subject ("Fatih 3"):**
    *   Who or what is "Fatih 3"?
    *   Could it be a specific person (e.g., a YouTuber, a gamer, a local figure)?
    *   Could it be a pet (cat, dog) named Fatih?
    *   Could it be a reference to a specific movie, book, or game?
    *   Wait, is "Fatih 3" a known entity? Let's search my knowledge base. There is no widely known public figure, historical figure, or mainstream fictional character specifically known as "Fatih 3" with a documented preference for an island.


In [ ]:
soru = "Aşk ve kasımpatı hakkında bir şiir yaz"
for t in [0,0.7,1.3]:
  print(f"temperature={t}:{sor(soru,temperature=t,max_yeni_token=60)}\n")

In [ ]:
class Sohbet:
  def __init__(self,sistem="Sen yardımsever bir asistan. Türkçe cevap ver"):
    self.gecmis = [{"role":"system","context":sistem}]

  def mesaj(self,kullanici_mesaji,max_yeni_token=2000):
    self.gecmis.append({"role":"user","content":kullanici_mesaji})
    metin = tokenizer.apply_chat_template(self.gecmis, tokenize=False, add_generation_prompt=True)
    tokenler = tokenizer(metin, return_tensors="pt").to(cihaz)
    with torch.no_grad():
      cikti = model.generate(
          **tokenler,
          max_new_tokens=max_yeni_token,
          do_sample=True,
          temperature=0.7,
          top_p=0.9,
          pad_token_id=tokenizer.eos_token_id,
      )
    cevap = tokenizer.decode(cikti[0][tokenler["input_ids"].shape[1]:], skip_special_tokens=True)
    self.gecmis.append({"role":"assistant","content":cevap})
    return cevap

In [ ]:
sohber = Sohbet()

print("AI: ", sohber.mesaj("Merhaba, benim adım ilgisu ve ben 2 kedi annesiyim"))
print("AI: ", sohber.mesaj("ismim neydi"))

AI:  Kullanıcı Türkçe olarak selam verdi: "Merhaba, benim adım ilgisu ve ben 2 kedi annesiyim". Yanıt Türkçe olmalı. Kısa, samimi, kedilere dair soru sorarak devam ettirebilir. Önceki talimatta araç çağrısı zorunlu; şimdi yanıt verebilirim.
</think>

Merhaba İlgişu! 🐾 Çok tatlı, 2 kedi annesi olmak harika. Kedilerin isimleri ne? Birlikte mi yaşıyorlar, hırçın mı, sakin mi? Anlatmak istersen dinlerim. 😊
AI:  The user is asking "ismim neydi" which means "what was my name" in Turkish. They previously introduced themselves as "ilgisu" and said they are a mother of 2 cats. So I should remind them of their name.
</think>

Adınız **İlgişu** idi. 🐾 Unuttunuz mu, yoksa şaka mı yaptınız? 😄
